# RakshAI — Baseline Donor Engagement Classification
### Google Colab Reproducible Experiment Notebook (Task 05)

This notebook trains and evaluates privacy-safe baseline machine learning models to classify donor activity status (`target_active`: 1 = Active, 0 = Inactive) using 5-Fold `GroupKFold` cross-validation.

**Core Methodological Safeguards:**
1. **Zero Data Leakage:** All deterministic business rule features (`inactive_trigger_comment`, `last_donation_date_days`, `total_calls`, `calls_to_donations_ratio`) and identifiers have been completely excluded.
2. **Grouped Cross-Validation:** Folds are partitioned strictly on `donor_group_id` using `GroupKFold(n_splits=5)` so multiple records from the same donor never span both train and test splits.
3. **In-Fold Preprocessing:** Feature scalers, imputers, and encoders are fit strictly on training folds via `Pipeline` and `ColumnTransformer` to eliminate test leakage.
4. **Class Imbalance Awareness:** Evaluates on PR-AUC, minority Inactive-class F1, and balanced accuracy rather than naive accuracy.

In [ ]:
# 1. Environment & Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    balanced_accuracy_score, average_precision_score, roc_auc_score,
    confusion_matrix, classification_report
)

print("Libraries loaded successfully.")

In [ ]:
# 2. Dataset Loading
# When running in Colab, upload 'rakshai_privacy_safe_dataset.csv' via the sidebar
data_path = 'rakshai_privacy_safe_dataset.csv'
df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
print(f"Unique Donor Groups: {df['donor_group_id'].nunique()}")
display(df.head())

# Target distribution
counts = df['target_active'].value_counts()
pcts = df['target_active'].value_counts(normalize=True) * 100
print("\n--- Target Distribution ---")
print(f"Active (1):   {counts[1]} ({pcts[1]:.2f}%)")
print(f"Inactive (0): {counts[0]} ({pcts[0]:.2f}%)")
print(f"Majority Class Baseline Accuracy: {pcts[1]:.2f}%")

In [ ]:
# 3. Feature Group Definitions
core_num_cols = ['cycle_of_donations', 'frequency_in_days', 'registration_days']
core_cat_cols = ['role', 'blood_group', 'gender', 'donor_type', 'eligibility_status']

bridge_num_cols = ['quantity_required']
bridge_cat_cols = ['bridge_gender', 'bridge_blood_group']

y = df['target_active'].values
groups = df['donor_group_id'].values

print(f"Core Features: {len(core_num_cols)} numeric + {len(core_cat_cols)} categorical")
print(f"Bridge Features: {len(bridge_num_cols)} numeric + {len(bridge_cat_cols)} categorical")

In [ ]:
# 4. Preprocessor Pipeline Builder Function
def build_preprocessor(num_cols, cat_cols):
    num_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    
    cat_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, num_cols),
            ('cat', cat_transformer, cat_cols)
        ]
    )
    return preprocessor

In [ ]:
# 5. 5-Fold GroupKFold Cross-Validation Evaluation Engine
def evaluate_model_group_cv(model, X_df, y, groups, n_splits=5):
    gkf = GroupKFold(n_splits=n_splits)
    
    metrics = {
        'inactive_precision': [],
        'inactive_recall': [],
        'inactive_f1': [],
        'macro_f1': [],
        'balanced_acc': [],
        'pr_auc_inactive': [],
        'roc_auc': []
    }
    
    oof_preds = np.zeros(len(y))
    oof_probs = np.zeros(len(y))
    
    # Target for inactive class evaluation (inactive = 1 for precision-recall focus)
    y_inactive = (y == 0).astype(int)
    
    for fold, (train_idx, test_idx) in enumerate(gkf.split(X_df, y, groups=groups)):
        X_train, X_test = X_df.iloc[train_idx], X_df.iloc[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        y_test_inact = y_inactive[test_idx]
        
        # Fit pipeline inside training fold only
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        oof_preds[test_idx] = preds
        
        # Predict probabilities
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(X_test)
            # prob of active
            prob_active = probs[:, 1]
            # prob of inactive
            prob_inactive = probs[:, 0]
            oof_probs[test_idx] = prob_inactive
            
            pr_auc = average_precision_score(y_test_inact, prob_inactive)
            roc_auc = roc_auc_score(y_test, prob_active)
        else:
            pr_auc = 0.0
            roc_auc = 0.5
            
        # Inactive class is 0
        inact_prec = precision_score(y_test, preds, pos_label=0, zero_division=0)
        inact_rec = recall_score(y_test, preds, pos_label=0, zero_division=0)
        inact_f1 = f1_score(y_test, preds, pos_label=0, zero_division=0)
        macro_f1 = f1_score(y_test, preds, average='macro', zero_division=0)
        bal_acc = balanced_accuracy_score(y_test, preds)
        
        metrics['inactive_precision'].append(inact_prec)
        metrics['inactive_recall'].append(inact_rec)
        metrics['inactive_f1'].append(inact_f1)
        metrics['macro_f1'].append(macro_f1)
        metrics['balanced_acc'].append(bal_acc)
        metrics['pr_auc_inactive'].append(pr_auc)
        metrics['roc_auc'].append(roc_auc)
        
    return metrics, oof_preds, oof_probs

In [ ]:
# 6. Execution: Baselines & Logistic Regression Experiments

# Experiment 1: Majority-Class Baseline (Always Active)
dummy_maj = DummyClassifier(strategy='most_frequent')
m_maj, _, _ = evaluate_model_group_cv(dummy_maj, df[core_num_cols + core_cat_cols], y, groups)

# Experiment 2: Stratified Random Baseline
dummy_strat = DummyClassifier(strategy='stratified', random_state=42)
m_strat, _, _ = evaluate_model_group_cv(dummy_strat, df[core_num_cols + core_cat_cols], y, groups)

# Experiment 3: Regularized Logistic Regression (Core Features Only)
prep_core = build_preprocessor(core_num_cols, core_cat_cols)
pipe_lr_core = Pipeline([
    ('prep', prep_core),
    ('clf', LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=42))
])
m_lr_core, oof_preds_core, oof_probs_core = evaluate_model_group_cv(
    pipe_lr_core, df[core_num_cols + core_cat_cols], y, groups
)

# Experiment 4: Regularized Logistic Regression (Core + Bridge Features)
all_num = core_num_cols + bridge_num_cols
all_cat = core_cat_cols + bridge_cat_cols
prep_all = build_preprocessor(all_num, all_cat)
pipe_lr_all = Pipeline([
    ('prep', prep_all),
    ('clf', LogisticRegression(class_weight='balanced', C=0.1, max_iter=1000, random_state=42))
])
m_lr_all, oof_preds_all, oof_probs_all = evaluate_model_group_cv(
    pipe_lr_all, df[all_num + all_cat], y, groups
)

print("Evaluation complete.")

In [ ]:
# 7. Results Comparison Summary Table
results_summary = pd.DataFrame([
    {
        'Model': 'Majority-Class Baseline',
        'Balanced Acc': np.mean(m_maj['balanced_acc']),
        'Inactive Precision': np.mean(m_maj['inactive_precision']),
        'Inactive Recall': np.mean(m_maj['inactive_recall']),
        'Inactive F1': np.mean(m_maj['inactive_f1']),
        'Macro F1': np.mean(m_maj['macro_f1']),
        'Inactive PR-AUC': np.mean(m_maj['pr_auc_inactive'])
    },
    {
        'Model': 'Stratified Random Baseline',
        'Balanced Acc': np.mean(m_strat['balanced_acc']),
        'Inactive Precision': np.mean(m_strat['inactive_precision']),
        'Inactive Recall': np.mean(m_strat['inactive_recall']),
        'Inactive F1': np.mean(m_strat['inactive_f1']),
        'Macro F1': np.mean(m_strat['macro_f1']),
        'Inactive PR-AUC': np.mean(m_strat['pr_auc_inactive'])
    },
    {
        'Model': 'LogReg (Core Features)',
        'Balanced Acc': np.mean(m_lr_core['balanced_acc']),
        'Inactive Precision': np.mean(m_lr_core['inactive_precision']),
        'Inactive Recall': np.mean(m_lr_core['inactive_recall']),
        'Inactive F1': np.mean(m_lr_core['inactive_f1']),
        'Macro F1': np.mean(m_lr_core['macro_f1']),
        'Inactive PR-AUC': np.mean(m_lr_core['pr_auc_inactive'])
    },
    {
        'Model': 'LogReg (Core + Bridge Features)',
        'Balanced Acc': np.mean(m_lr_all['balanced_acc']),
        'Inactive Precision': np.mean(m_lr_all['inactive_precision']),
        'Inactive Recall': np.mean(m_lr_all['inactive_recall']),
        'Inactive F1': np.mean(m_lr_all['inactive_f1']),
        'Macro F1': np.mean(m_lr_all['macro_f1']),
        'Inactive PR-AUC': np.mean(m_lr_all['pr_auc_inactive'])
    }
]).set_index('Model')

display(results_summary.round(4))

# Confusion Matrix for Core Logistic Regression
cm = confusion_matrix(y, oof_preds_core)
print("\n--- Out-of-Fold Confusion Matrix (Core LogReg) ---")
print("                 Predicted Inactive (0)   Predicted Active (1)")
print(f"Actual Inactive (0):      {cm[0, 0]:<20}    {cm[0, 1]}")
print(f"Actual Active   (1):      {cm[1, 0]:<20}    {cm[1, 1]}")

## 8. Clinical & Methodological Limitations

1. **Retrospective Administrative Label:** `target_active` reflects a historical database status snapshot rather than a prospective 90-day clinical response event. Inactive donors were marked based on prolonged non-donation or lack of response to phone outreach.
2. **Medical Safety Principle:** Machine learning models should **only** be utilized to assist with voluntary donor re-engagement and prioritization. Under clinical protocols, blood group compatibility (ABO/Rh), donation interval compliance (90 days for males / 120 days for females), and minor antigen alloimmunization safety must always be enforced via deterministic hematological verification rules.